<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Cleans the raw grade lookup table and normalizes IDs for joins.

**Notebook Shape:** 42 cells (26 code, 16 markdown).

**Inputs / Data Sources:**
- `df_raw = pd.read_parquet(RAW_DIR / "v_acs_grade.parquet")`

**Outputs / Side Effects:**
- `df_clean_grade.to_parquet(PREPROCESSED_DIR / "V_ACS_GRADE" / "clean_v_acs_grade.parquet", index=False)`

**Logic Flow:**
1. Load raw ACS grade parquet.
2. Normalize grade and ID columns.
3. Inspect duplicates and type quality.
4. Write the cleaned grade parquet.

**Maintainability Notes:** Grade semantics feed target interpretation; keep cleaning assumptions documented beside any target-construction code.

# ACS_grade - Cleaning Notebook

This notebook cleans the `ACS_grade` lookup table.

The clean output is `df_clean_grade`.

This table is used later as a grade lookup by `grade_id` and as a grading range lookup by `grade_version_id`, `from_percent`, and `to_percent`.

This notebook only performs cleaning and validation. It does not train models, create ML features, encode categories, scale numeric values, or save files.

## Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.id_casting import normalize_ids
from src.schemas import TABLE_ACS_GRADE
from src.paths import RAW_DIR, PREPROCESSED_DIR, assert_data_root
from src.io_utils import save_parquet

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

## Configuration

In [2]:
# Fill this manually only if you want this notebook to load the parquet file.
# If df_raw is already loaded, leave this empty and skip the optional loading cell.
DATA_PATH = RAW_DIR / "v_acs_grade.parquet"

# Data-root guard (governance contract 12).
assert_data_root(DATA_PATH)

## Optional Loading Cell

In [3]:
# Run this cell only after filling DATA_PATH, or skip it if df_raw is already loaded.
if DATA_PATH:
    df_raw = pd.read_parquet(Path(DATA_PATH))
else:
    print("DATA_PATH is empty. Fill DATA_PATH or provide df_raw manually before preprocessing.")

## Preprocessing

In [4]:
df = df_raw.copy()
original_shape = df.shape

df.columns = (
    pd.Index(df.columns)
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

In [5]:
required_columns = [
    "grade_id",
    "grade_version_id",
    "grade_name_sl",
    "version_title_sl",
    "version_number",
    "from_percent",
    "to_percent",
    "points",
    "finish_status",
    "grade_show",
    "active",
]

missing_columns = [column for column in required_columns if column not in df.columns]
assert not missing_columns, f"Missing required columns: {missing_columns}"

In [6]:
string_columns = [
    "grade_name_sl",
    "version_title_sl",
    "version_number",
    "finish_status",
    "grade_show",
    "active",
]

for column in string_columns:
    cleaned_text = df[column].astype("string").str.strip()
    df[column] = cleaned_text.mask(cleaned_text.str.lower().isin(["", "nan", "none", "null"]), pd.NA)

df["finish_status"] = df["finish_status"].str.upper()

In [7]:
# In-place ID casting per src/schemas.py: grade_id, grade_version_id -> string.
# No _key duplicates.
df = normalize_ids(df, table=TABLE_ACS_GRADE)

In [8]:
for column in ["from_percent", "to_percent", "points"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.drop(columns=["active"])

## Finish Status Filter

In [9]:
allowed_finish_statuses = {"P", "F", "FA", "FE"}

bad_finish_status_mask = df["finish_status"].isna() | ~df["finish_status"].isin(allowed_finish_statuses)
dropped_bad_finish_status_report = df.loc[bad_finish_status_mask].copy()
df = df.loc[~bad_finish_status_mask].copy()

## Critical Nulls

In [10]:
critical_columns = [
    "grade_id",
    "grade_version_id",
    "version_number",
    "from_percent",
    "to_percent",
    "points",
    "finish_status",
]

critical_null_mask = df[critical_columns].isna().any(axis=1)
dropped_critical_null_report = df.loc[critical_null_mask].copy()
df = df.loc[~critical_null_mask].copy()

## Duplicate Handling

In [11]:
exact_duplicate_mask = df.duplicated(keep=False)
exact_duplicate_report = df.loc[exact_duplicate_mask].copy()
exact_duplicate_rows_dropped = int(df.duplicated(keep="first").sum())
df = df.drop_duplicates().copy()

duplicate_grade_id_report = (
    df.loc[df["grade_id"].duplicated(keep=False)]
    .sort_values("grade_id")
    .copy()
)

## Numeric Validation Reports

In [12]:
invalid_percent_range_report = df.loc[
    df["from_percent"].lt(0)
    | df["to_percent"].gt(100)
    | df["from_percent"].gt(df["to_percent"])
].copy()

invalid_points_report = df.loc[
    df["points"].lt(0)
    | df["points"].gt(4)
].copy()

## Range and Version Validation Reports

In [13]:
range_issues = []

for grade_version_id, version_df in df.sort_values(["grade_version_id", "from_percent", "to_percent"]).groupby("grade_version_id", dropna=False):
    previous_row = None

    for _, current_row in version_df.iterrows():
        if previous_row is not None:
            if current_row["from_percent"] <= previous_row["to_percent"]:
                range_issues.append(
                    {
                        "grade_version_id": grade_version_id,
                        "version_number": current_row["version_number"],
                        "issue_type": "overlap",
                        "previous_from_percent": previous_row["from_percent"],
                        "previous_to_percent": previous_row["to_percent"],
                        "current_from_percent": current_row["from_percent"],
                        "current_to_percent": current_row["to_percent"],
                        "message": "Current inclusive range starts before or at the previous range end.",
                    }
                )
            elif current_row["from_percent"] > previous_row["to_percent"] + 1:
                range_issues.append(
                    {
                        "grade_version_id": grade_version_id,
                        "version_number": current_row["version_number"],
                        "issue_type": "gap",
                        "previous_from_percent": previous_row["from_percent"],
                        "previous_to_percent": previous_row["to_percent"],
                        "current_from_percent": current_row["from_percent"],
                        "current_to_percent": current_row["to_percent"],
                        "message": "Current inclusive range starts more than 1 point after the previous range end.",
                    }
                )

        previous_row = current_row

grade_range_validation_report = pd.DataFrame(
    range_issues,
    columns=[
        "grade_version_id",
        "version_number",
        "issue_type",
        "previous_from_percent",
        "previous_to_percent",
        "current_from_percent",
        "current_to_percent",
        "message",
    ],
)

In [14]:
version_consistency_report = (
    df[["grade_version_id", "version_number", "version_title_sl"]]
    .drop_duplicates()
    .sort_values(["grade_version_id", "version_number", "version_title_sl"])
    .reset_index(drop=True)
)

version_number_counts = (
    version_consistency_report.groupby("grade_version_id", dropna=False)["version_number"]
    .nunique()
    .rename("version_number_count")
    .reset_index()
)

bad_version_mapping_report = (
    version_consistency_report.merge(
        version_number_counts.loc[version_number_counts["version_number_count"].gt(1)],
        on="grade_version_id",
        how="inner",
    )
    .sort_values(["grade_version_id", "version_number"])
    .reset_index(drop=True)
)

## Final Table

In [15]:
final_columns = [
    "grade_id",
    "grade_version_id",
    "version_number",
    "version_title_sl",
    "from_percent",
    "to_percent",
    "points",
    "finish_status",
    "grade_show",
    "grade_name_sl",
]

df_clean_grade = (
    df[final_columns]
    .sort_values(["grade_version_id", "from_percent", "to_percent", "grade_id"])
    .reset_index(drop=True)
)

## Assertions

In [16]:
assert not missing_columns, f"Missing required columns: {missing_columns}"
assert df_clean_grade["grade_id"].is_unique, "grade_id must be unique. See duplicate_grade_id_report."
assert not df_clean_grade[critical_columns].isna().any().any(), "Critical columns contain nulls. See dropped_critical_null_report."
assert invalid_percent_range_report.empty, "Invalid percent ranges found. See invalid_percent_range_report."
assert invalid_points_report.empty, "Invalid GPA points found. See invalid_points_report."
assert set(df_clean_grade["finish_status"].dropna()).issubset(allowed_finish_statuses), "Unexpected finish_status values found."
assert "active" not in df_clean_grade.columns, "active column must be removed from df_clean_grade."

## Summary

In [17]:
summary = {
    "original_shape": original_shape,
    "final_shape": df_clean_grade.shape,
    "exact_duplicate_rows_dropped": exact_duplicate_rows_dropped,
    "bad_or_null_finish_status_rows_dropped": len(dropped_bad_finish_status_report),
    "critical_null_rows_dropped": len(dropped_critical_null_report),
    "number_of_grade_versions": df_clean_grade["grade_version_id"].nunique(dropna=True),
    "final_columns": df_clean_grade.columns.tolist(),
    "grade_id_is_unique": df_clean_grade["grade_id"].is_unique,
}

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nfinal finish_status value counts:")
print(df_clean_grade["finish_status"].value_counts(dropna=False))

original_shape: (72, 11)
final_shape: (40, 10)
exact_duplicate_rows_dropped: 0
bad_or_null_finish_status_rows_dropped: 32
critical_null_rows_dropped: 0
number_of_grade_versions: 3
final_columns: ['grade_id', 'grade_version_id', 'version_number', 'version_title_sl', 'from_percent', 'to_percent', 'points', 'finish_status', 'grade_show', 'grade_name_sl']
grade_id_is_unique: True

final finish_status value counts:
finish_status
P     31
FE     3
FA     3
F      3
Name: count, dtype: int64[pyarrow]


## Outputs Created

This notebook creates these in-memory objects only:

- `df_clean_grade`
- `exact_duplicate_report`
- `dropped_bad_finish_status_report`
- `dropped_critical_null_report`
- `duplicate_grade_id_report`
- `invalid_percent_range_report`
- `invalid_points_report`
- `grade_range_validation_report`
- `version_consistency_report`
- `bad_version_mapping_report`

No parquet files or external outputs are saved.

In [18]:
df2=df_clean_grade.sort_values(["grade_version_id", "from_percent"])[
        [
            "grade_id",
            "grade_version_id",
            "version_number",
            "from_percent",
            "to_percent",
            "points",
            "finish_status",
            "grade_show",
            "grade_name_sl"
        ]
    ]


In [19]:
df2

,grade_id,grade_version_id,version_number,from_percent,to_percent,points,finish_status,grade_show,grade_name_sl
0,1003.111,1.111,1,0,0,0.00,FE,F,راسب امتحان نهائي
1,96.111,1.111,1,0,0,0.00,FA,Z,محروم بالغياب
2,94.111,1.111,1,0,59,0.00,F,F,رسوب
3,93.111,1.111,1,60,66,1.00,P,D,مقبول
4,92.111,1.111,1,67,69,1.70,P,C-,جيد
5,91.111,1.111,1,70,72,2.00,P,C,جيد
6,90.111,1.111,1,73,76,2.30,P,C+,جيد
7,89.111,1.111,1,77,79,2.70,P,B-,جيد جداً
8,88.111,1.111,1,80,81,3.00,P,B,جيد جداً
9,87.111,1.111,1,82,86,3.30,P,B+,جيد جداً


In [20]:
save_parquet(df_clean_grade, PREPROCESSED_DIR / "V_ACS_GRADE" / "clean_v_acs_grade.parquet")

WindowsPath('D:/AI/Real projects/Academic_Advisor/data/preprocessed/V_ACS_GRADE/clean_v_acs_grade.parquet')

In [21]:
df_clean_grade.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   grade_id          40 non-null     string 
 1   grade_version_id  40 non-null     string 
 2   version_number    40 non-null     string 
 3   version_title_sl  40 non-null     string 
 4   from_percent      40 non-null     int64  
 5   to_percent        40 non-null     int64  
 6   points            40 non-null     float64
 7   finish_status     40 non-null     string 
 8   grade_show        40 non-null     string 
 9   grade_name_sl     40 non-null     string 
dtypes: float64(1), int64(2), string(7)
memory usage: 5.6 KB


In [22]:
df_clean_grade.head()

,grade_id,grade_version_id,version_number,version_title_sl,from_percent,to_percent,points,finish_status,grade_show,grade_name_sl
0,1003.111,1.111,1,تأسيس الجامعة,0,0,0.0,FE,F,راسب امتحان نهائي
1,96.111,1.111,1,تأسيس الجامعة,0,0,0.0,FA,Z,محروم بالغياب
2,94.111,1.111,1,تأسيس الجامعة,0,59,0.0,F,F,رسوب
3,93.111,1.111,1,تأسيس الجامعة,60,66,1.0,P,D,مقبول
4,92.111,1.111,1,تأسيس الجامعة,67,69,1.7,P,C-,جيد


In [23]:
import sys
print(sys.executable)

d:\AI\Real projects\Academic_Advisor\.venv\Scripts\python.exe


In [24]:
%pip install ipykernel notebook

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
